# Exploring the data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/RRCA_baseflow.csv')

# Date is a matlab day number
df['DateTime'] = pd.to_datetime(df['Date'] - 719529, unit='D')
df['DateTime'].head(10)

Jan 29 shows up right after Jan 1 so some of these are actually the next month. Rounding to the nearest month fixes it.

In [ ]:
df['DateTime'] = (df['DateTime'] + pd.Timedelta(days=15)).dt.to_period('M').dt.to_timestamp()
df['Year'] = df['DateTime'].dt.year
df['Month'] = df['DateTime'].dt.month

# pumping is stored as negative numbers
df['Pumping'] = df['Irrigation_pumping'] * -1

display(df.head(5))
df.describe()

## Segments

There are duplicate segment/date rows, checking why

In [ ]:
df.groupby('Segment_id')[['x', 'y']].nunique().sort_values('x', ascending=False).head()

In [ ]:
df[df['Segment_id'] == 53].groupby(['x', 'y'])['Observed'].agg(['count', 'mean'])

Some segments have 2 gauges at different spots with really different flows, so I'm making a station id from x/y instead of using Segment_id.

In [ ]:
df['Station'] = df.groupby(['x', 'y']).ngroup()
df['Station'].nunique()

In [ ]:
df.groupby('Station').agg({'Segment_id': 'first', 'Year': ['min', 'max'], 'Observed': 'count'})

## Zeros

In [ ]:
(df['Precipitation'] == 0).mean()

In [ ]:
df.groupby('Station')['Precipitation'].apply(lambda x: (x == 0).mean()).sort_values(ascending=False).head(12)

In [ ]:
df[df['Station'] == 0][['Precipitation', 'Evapotranspiration', 'Pumping']].describe()

In [ ]:
df[(df['Station'] == 38) & (df['Precipitation'] == 0)]['Year'].unique()

9 stations have 0 for precipitation, ET, and pumping the entire time so that's missing data not real zeros. Station 38 loses its precipitation after 1994.

Also precipitation looks like the same number every month of a year:

In [ ]:
df[df['Station'] == 15][['DateTime', 'Precipitation', 'Evapotranspiration']].head(15)

So precipitation is yearly. A whole year with 0 rain isn't realistic so I'm dropping rows where precipitation is 0.

In [ ]:
df = df[df['Precipitation'] > 0]
print(len(df), df['Station'].nunique())

## Baseflow

In [ ]:
sns.histplot(df['Observed'], bins=100)
plt.show()

In [ ]:
df[df['Observed'] < 0]['Station'].value_counts()

Super skewed, most is under 25 but a few go over 700. Negatives are almost all station 9, which is probably a losing stream (river leaking into the ground). Keeping them since they're real.

## Map

In [ ]:
station_means = df.groupby('Station')[['x', 'y', 'Observed', 'Pumping']].mean()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=station_means, x='x', y='y', hue='Observed', size='Observed')
plt.title('avg baseflow')
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=station_means, x='x', y='y', hue='Pumping', size='Pumping')
plt.title('avg pumping')
plt.show()

The two biggest baseflow stations are on the east end (downstream, river flows into Kansas). Most pumping is on the west side.

## Over time

In [ ]:
yearly = df.groupby('Year')[['Observed', 'Pumping', 'Precipitation', 'Evapotranspiration']].mean()
yearly.plot(subplots=True, figsize=(10, 9))
plt.show()

Baseflow drops a ton around 1950 but I think that's just some big stations stop getting recorded then. Pumping starts in the mid 50s and goes up until about 1980. Rain and ET don't really trend.

## Looking at single stations

In [ ]:
station_means.sort_values('Pumping', ascending=False).head()

Picking station 7 (lots of pumping and a long record), station 4 (not much pumping), and station 9 (the losing stream).

In [ ]:
for s in [7, 4, 9]:
    d = df[df['Station'] == s]
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(d['DateTime'], d['Observed'])
    ax.set_ylabel('baseflow')
    ax2 = ax.twinx()
    ax2.plot(d['DateTime'], d['Pumping'], color='red')
    ax2.set_ylabel('pumping', color='red')
    plt.title('station ' + str(s))
    plt.show()

Pumping only happens in summer, so it's gonna be tied up with month. Station 7 slowly drops after pumping starts in the 50s. Station 9 starts going negative in the summers once pumping picks up.

In [ ]:
corrs = []
for s in df['Station'].unique():
    d = df[df['Station'] == s]
    if d['Pumping'].max() > 0:
        corrs.append([s, d['Observed'].corr(d['Pumping'])])

corrs = pd.DataFrame(corrs, columns=['Station', 'corr']).sort_values('corr')
corrs.plot(x='Station', y='corr', kind='bar', figsize=(12, 4), legend=False)
plt.title('baseflow vs pumping correlation for each station')
plt.show()

Inside basically every station more pumping = less baseflow. Only station 17 goes the other way.

## Baseflow vs everything

In [ ]:
sns.scatterplot(data=df, x='Pumping', y='Observed', alpha=0.2, s=10)
plt.show()
sns.scatterplot(data=df, x='Precipitation', y='Observed', alpha=0.2, s=10)
plt.show()
sns.scatterplot(data=df, x='Evapotranspiration', y='Observed', alpha=0.2, s=10)
plt.show()

In [ ]:
sns.boxplot(data=df, x='Month', y='Observed', showfliers=False)
plt.show()

In [ ]:
sns.heatmap(df[['Observed', 'Pumping', 'Precipitation', 'Evapotranspiration', 'x', 'y']].corr(), annot=True, cmap='coolwarm')
plt.show()

Correlations over everything are pretty weak. Baseflow is lowest Jul-Sep which is when pumping happens. Pumping and ET are correlated (0.46) since they're both high in summer.